## **Default Chat Model Setup**

In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

google_model="gemini-3.1-flash-lite"

In [2]:
from langchain.chat_models import init_chat_model

llm_gemini = init_chat_model(model=google_model,model_provider="openai",
    openai_api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

## **Guiding in Prompts**

In [3]:
llm_gemini.invoke("Write a joke about cars in a genz tone. Generate a joke in a key-value format with the keys: setup and punchline")

AIMessage(content='{\n  "setup": "Why did the Gen Z driver break up with their car?",\n  "punchline": "Because it had zero rizz and kept ghosting them at every red light."\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 29, 'total_tokens': 73, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemini-3.1-flash-lite', 'system_fingerprint': None, 'id': 'IShWao-IKPj0xN8P9PKfcA', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f608c-c312-7650-8ad8-241a42ec5096-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {}, 'output_token_details': {}})

## **Using Pydantic Schema**

In [4]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(..., description="The setup of the joke")
    punchline: str = Field(..., description="The punchline of the joke")


In [5]:
obj = llm_schema(**{"setup": "some setup", "punchline": "some punchline"})
obj

llm_schema(setup='some setup', punchline='some punchline')

In [6]:
llm_structured_output = llm_gemini.with_structured_output(llm_schema)
llm_structured_output

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-openai': '1.3.5'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x1174a4b30>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11790b1d0>, root_client=<openai.OpenAI object at 0x112db8b00>, root_async_client=<openai.AsyncOpenAI object at 0x1202e3830>, model_name='gemini-3.1-flash-lite', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://generativelanguage.googleapis.com/v1beta/openai/', openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.llm_schema'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'llm_schema', 'description': '', 'parameters': {'properties': {'setup': {'description': 'The setup of the joke', 'type': 

In [7]:
llm_structured_output.invoke("Write a joke about cars in a genz tone.")

llm_schema(setup='Why did the Gen Z driver stop their car in the middle of the intersection?', punchline='They saw a vibe check and decided to pull over to let it pass.')

In [8]:
response = llm_structured_output.invoke("Write a joke about cars in a genz tone.")
# won't work with Pydantic models, but will work with TypedDicts
# response["punchline"]

# will work with Pydantic models, and will have type checking
response.punchline

'Because their engine was literally ghosting them.'

## **Using TypedDict Schema**

In [9]:
from typing import TypedDict

class llm_schema_td(TypedDict):
    setup: str
    punchline: str

In [10]:
obj = llm_schema_td(**{"setup": "some setup", "punchline": "some punchline"})
obj

{'setup': 'some setup', 'punchline': 'some punchline'}

In [11]:
# Also works with TypedDicts. Not strict type checking, but useful for type hints.
faulty_obj = llm_schema_td(**{"ketchup": "some setup", "punchline": "some punchline"})
faulty_obj

{'ketchup': 'some setup', 'punchline': 'some punchline'}

In [12]:
llm_structured_output_td = llm_gemini.with_structured_output(llm_schema_td)
llm_structured_output_td

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-openai': '1.3.5'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x1174a4b30>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11790b1d0>, root_client=<openai.OpenAI object at 0x112db8b00>, root_async_client=<openai.AsyncOpenAI object at 0x1202e3830>, model_name='gemini-3.1-flash-lite', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://generativelanguage.googleapis.com/v1beta/openai/', openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'response_format': {'type': 'json_schema', 'json_schema': {'name': 'llm_schema_td', 'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n    (key, value) pairs\ndict(iterable) -> new dictionary initialized as if via:\n    d = {}\n    for k, v i

In [13]:
llm_structured_output_td.invoke("Write a joke about cars in a genz tone.")

{'setup': 'Why did the Gen Z driver pull over their car?',
 'punchline': 'Because the vibe check failed and it was giving major check engine light energy.'}

In [14]:
response = llm_structured_output_td.invoke("Write a joke about cars in a genz tone.")
# won't work with TypedDicts, but will work with Pydantic models
# response.punchline

# will work with TypedDicts, but will not have type checking
response["punchline"]

'Because it literally could not handle their energy and went into sleep mode.'